In [2]:
# -*- coding: utf-8 -*-
"""
TUGAS PRAKTIKUM - BLOCKCHAIN UNTUK BUKTI DIGITAL
NIM: 23040700004
NAMA: MUHAMMAD SULTAN ARIF
"""

# ============================================
# 1. INSTALL & IMPORT DEPENDENSI
# ============================================
import subprocess
import sys
import os
import hashlib
import zipfile
import shutil
from datetime import datetime

# Install Scapy
try:
    import scapy
    print("✅ Scapy sudah terinstall")
except ImportError:
    print("📦 Menginstall Scapy...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scapy", "-q"])
    print("✅ Scapy berhasil diinstall")

# Import dari Scapy
from scapy.all import IP, TCP, UDP, ICMP, Ether, wrpcap, rdpcap, RandIP, RandShort

# ============================================
# 2. KONFIGURASI
# ============================================
NIM = "23040700004"
NAMA = "MUHAMMAD SULTAN ARIF"

print("="*80)
print(f"🔐 TUGAS PRAKTIKUM - BLOCKCHAIN BUKTI DIGITAL")
print(f"👤 NIM: {NIM}")
print(f"👤 Nama: {NAMA}")
print("="*80)

# ============================================
# 3. AUTO GENERATE PCAP FILES
# ============================================
print("\n📡 STEP 1: MEMBUAT 5 FILE PCAP SAMPLE")
print("="*80)

def create_pcap_files(nim):
    """Membuat 5 file PCAP dengan jumlah paket berbeda"""

    pcap_configs = [
        (30, f"PCAP01_{nim}.pcapng"),
        (50, f"PCAP02_{nim}.pcapng"),
        (70, f"PCAP03_{nim}.pcapng"),
        (90, f"PCAP04_{nim}.pcapng"),
        (100, f"PCAP05_{nim}.pcapng")
    ]

    created_files = []

    print("\n📝 Membuat file PCAP sample...")
    print("-"*50)

    for count, filename in pcap_configs:
        print(f"\n📁 Membuat {filename} dengan {count} paket...")

        packets = []
        for i in range(count):
            pkt_type = i % 4

            if pkt_type == 0:
                pkt = Ether()/IP(src=RandIP(), dst=RandIP())/TCP(sport=RandShort(), dport=RandShort())
                pkt = pkt / f"TCP Packet #{i} - Sample data"
            elif pkt_type == 1:
                pkt = Ether()/IP(src=RandIP(), dst=RandIP())/UDP(sport=RandShort(), dport=RandShort())
                pkt = pkt / f"UDP Packet #{i} - Sample data"
            elif pkt_type == 2:
                pkt = Ether()/IP(src=RandIP(), dst=RandIP())/ICMP()
                pkt = pkt / f"ICMP Packet #{i} - Sample data"
            else:
                pkt = Ether()/IP(src=RandIP(), dst=RandIP())
                pkt = pkt / f"Raw Packet #{i} - Sample data"

            packets.append(pkt)

        wrpcap(filename, packets)
        created_files.append((filename, count))

        size = os.path.getsize(filename) / 1024
        print(f"   ✅ {filename}: {count} paket, {size:.2f} KB")

    print("\n" + "-"*50)
    print(f"✅ Berhasil membuat {len(created_files)} file PCAP!")

    return created_files

# Buat file PCAP
pcap_files = create_pcap_files(NIM)
pcap_filenames = [f[0] for f in pcap_files]

# ============================================
# 4. FUNGSI HITUNG HASH
# ============================================
def calculate_sha256(file_path):
    """Menghitung SHA-256 hash dari file"""
    sha256_hash = hashlib.sha256()
    try:
        with open(file_path, "rb") as f:
            for byte_block in iter(lambda: f.read(4096), b""):
                sha256_hash.update(byte_block)
        return sha256_hash.hexdigest()
    except Exception as e:
        print(f"❌ Error menghitung hash {file_path}: {e}")
        return None

def get_packet_count(pcap_file):
    """Mendapatkan jumlah paket dari file PCAP"""
    try:
        packets = rdpcap(pcap_file)
        return len(packets)
    except Exception as e:
        print(f"⚠️  Error membaca {pcap_file}: {e}")
        return 0

# ============================================
# 5. PROSES FILE PCAP
# ============================================
print("\n📊 STEP 2: PROSES FILE PCAP")
print("="*80)

results = []
for filename in sorted(pcap_filenames):
    packet_count = get_packet_count(filename)
    file_size = os.path.getsize(filename) / 1024
    hash_value = calculate_sha256(filename)

    results.append({
        'filename': filename,
        'packet_count': packet_count,
        'size_kb': file_size,
        'hash': hash_value
    })

    print(f"\n📁 {filename}")
    print(f"   📦 Paket: {packet_count}")
    print(f"   📊 Ukuran: {file_size:.2f} KB")
    print(f"   🔐 SHA-256: {hash_value}")

# ============================================
# 6. TAMPILKAN HASIL
# ============================================
print("\n" + "="*120)
print("📊 HASIL PERHITUNGAN SHA-256")
print("="*120)
print(f"\n{'No':<5} {'Nama File':<35} {'Paket':<12} {'Ukuran (KB)':<15} {'SHA-256 Hash'}")
print("-"*120)

for i, r in enumerate(results, 1):
    hash_display = r['hash'][:50] + "..." if r['hash'] else "ERROR"
    print(f"{i:<5} {r['filename']:<35} {r['packet_count']:<12} {r['size_kb']:<15.2f} {hash_display}")

print("="*120)

# ============================================
# 7. CLASS BLOCKCHAIN
# ============================================
class Blockchain:
    def __init__(self):
        self.chain = []
        self.create_genesis()

    def create_genesis(self):
        genesis = {
            'index': 0,
            'timestamp': datetime.now().isoformat(),
            'evidence_file': 'GENESIS BLOCK',
            'packet_count': 0,
            'evidence_hash': '0'*64,
            'previous_hash': '0'*64
        }
        genesis['block_hash'] = self.calc_hash(genesis)
        self.chain.append(genesis)
        print("✅ Genesis Block dibuat")

    def calc_hash(self, block):
        data = f"{block['index']}{block['timestamp']}{block['evidence_file']}{block['packet_count']}{block['evidence_hash']}{block['previous_hash']}"
        return hashlib.sha256(data.encode()).hexdigest()

    def add_block(self, evidence_file, packet_count, evidence_hash):
        prev = self.chain[-1]
        block = {
            'index': len(self.chain),
            'timestamp': datetime.now().isoformat(),
            'evidence_file': evidence_file,
            'packet_count': packet_count,
            'evidence_hash': evidence_hash,
            'previous_hash': prev['block_hash']
        }
        block['block_hash'] = self.calc_hash(block)
        self.chain.append(block)
        print(f"✅ Block {block['index']} ditambahkan: {evidence_file}")
        return block

    def validate(self):
        print("\n🔍 VALIDASI BLOCKCHAIN...")
        print("-"*60)
        all_valid = True
        for i, block in enumerate(self.chain):
            calc = self.calc_hash(block)
            if calc != block['block_hash']:
                print(f"❌ Block {i} INVALID - Hash tidak sesuai")
                all_valid = False
            elif i > 0 and block['previous_hash'] != self.chain[i-1]['block_hash']:
                print(f"❌ Block {i} INVALID - Previous hash mismatch")
                all_valid = False
            else:
                print(f"✅ Block {i} VALID")

        print("-"*60)
        if all_valid:
            print("✅ Blockchain Validation : VALID")
        else:
            print("❌ Blockchain Validation : INVALID")
        return all_valid

    def display(self):
        print("\n" + "="*100)
        print("🔗 BLOCKCHAIN - Bukti Digital Jaringan")
        print("="*100)
        for block in self.chain:
            print(f"\n📦 BLOCK #{block['index']}")
            print(f"   📁 File: {block['evidence_file']}")
            print(f"   📦 Paket: {block['packet_count']}")
            print(f"   🕐 Timestamp: {block['timestamp']}")
            print(f"   🔗 Block Hash: {block['block_hash'][:40]}...")
            print("-"*100)

    def modify_and_validate(self, file_path):
        """Bonus: Simulasi modifikasi 1 byte"""
        print("\n" + "="*80)
        print("🎯 BONUS: SIMULASI DETEKSI PERUBAHAN")
        print("="*80)

        if not os.path.exists(file_path):
            print(f"❌ File tidak ditemukan: {file_path}")
            return False

        print(f"📁 File target: {file_path}")

        backup = file_path + ".backup"
        os.rename(file_path, backup)

        try:
            with open(backup, 'rb') as f:
                data = bytearray(f.read())

            if len(data) > 0:
                original_byte = data[0]
                data[0] = (data[0] + 1) % 256
                with open(file_path, 'wb') as f:
                    f.write(data)

                print(f"✏️  File dimodifikasi: byte pertama {original_byte} → {data[0]}")

                new_hash = calculate_sha256(file_path)
                for block in self.chain:
                    if block['evidence_file'] == os.path.basename(file_path):
                        block['evidence_hash'] = new_hash
                        block['block_hash'] = self.calc_hash(block)
                        print(f"🔄 Block {block['index']} diperbarui dengan hash baru")
                        break

                print("\n🔍 Validasi setelah modifikasi:")
                is_valid = self.validate()

                if not is_valid:
                    print("\n🚨 PERUBAHAN TERDETEKSI! Blockchain menjadi INVALID")
                    print("✅ BONUS BERHASIL: Blockchain mendeteksi perubahan 1 byte!")

                os.rename(backup, file_path)
                print("\n♻️  File asli direstore")
                return is_valid

        except Exception as e:
            print(f"❌ Error: {e}")
            if os.path.exists(backup):
                os.rename(backup, file_path)
            return False

# ============================================
# 8. BANGUN BLOCKCHAIN
# ============================================
print("\n🔗 STEP 3: MEMBANGUN BLOCKCHAIN")
print("="*80)

blockchain = Blockchain()

for r in results:
    blockchain.add_block(
        r['filename'],
        r['packet_count'],
        r['hash']
    )

blockchain.display()

# ============================================
# 9. VALIDASI
# ============================================
print("\n🔍 STEP 4: VALIDASI BLOCKCHAIN")
print("="*80)

is_valid = blockchain.validate()

# ============================================
# 10. BONUS: MODIFIKASI FILE
# ============================================
if results and len(results) > 0:
    print("\n🎯 STEP 5: BONUS - DETEKSI PERUBAHAN")
    print("="*80)

    test_file = results[0]['filename']
    print(f"📁 Menguji file: {test_file}")
    blockchain.modify_and_validate(test_file)

# ============================================
# 11. RINGKASAN
# ============================================
print("\n" + "="*80)
print("📋 RINGKASAN HASIL")
print("="*80)

print(f"\n✅ NIM: {NIM}")
print(f"✅ Nama: {NAMA}")
print(f"✅ Total File PCAP: {len(pcap_filenames)}")
print(f"✅ Total Blocks: {len(blockchain.chain)} (1 Genesis + {len(results)} Evidence)")
print(f"✅ Status Validasi: {'VALID' if is_valid else 'INVALID'}")

# ============================================
# 12. DOWNLOAD HASIL
# ============================================
print("\n📥 STEP 6: DOWNLOAD HASIL")
print("="*80)

# Buat folder hasil
if not os.path.exists('hasil'):
    os.makedirs('hasil')

for f in pcap_filenames:
    if os.path.exists(f):
        shutil.copy(f, os.path.join('hasil', f))

# Buat zip file
with zipfile.ZipFile('hasil_tugas.zip', 'w') as zipf:
    for f in pcap_filenames:
        if os.path.exists(f):
            zipf.write(f, os.path.join('PCAP_Files', f))

    summary = f"""
========================================
HASIL TUGAS BLOCKCHAIN BUKTI DIGITAL
========================================
NIM: {NIM}
Nama: {NAMA}
Tanggal: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

HASIL SHA-256:
"""
    for r in results:
        summary += f"""
{r['filename']}
  - Paket: {r['packet_count']}
  - Ukuran: {r['size_kb']:.2f} KB
  - SHA-256: {r['hash']}
"""

    summary += f"""
STATUS VALIDASI: {'VALID' if is_valid else 'INVALID'}

BONUS DETEKSI PERUBAHAN: {'BERHASIL' if not is_valid else 'TIDAK TERDETEKSI'}
"""

    with open('summary.txt', 'w') as f:
        f.write(summary)
    zipf.write('summary.txt')

    print("✅ File PCAP dan summary ditambahkan ke ZIP")

# Download dari Google Colab
try:
    from google.colab import files
    print("\n📥 Mendownload hasil_tugas.zip...")
    files.download('hasil_tugas.zip')
except:
    print("\n⚠️  Tidak dapat mendownload otomatis. Silakan download manual.")

print("\n" + "="*80)
print("✅ TUGAS SELESAI!")
print("📁 File hasil: hasil_tugas.zip")
print("="*80)

# Tampilkan file yang tersedia
print("\n📁 FILE YANG TERSEDIA:")
for f in sorted(os.listdir()):
    if f.endswith('.pcapng'):
        size = os.path.getsize(f) / 1024
        print(f"   📄 {f} ({size:.2f} KB)")

✅ Scapy sudah terinstall
🔐 TUGAS PRAKTIKUM - BLOCKCHAIN BUKTI DIGITAL
👤 NIM: 23040700004
👤 Nama: MUHAMMAD SULTAN ARIF

📡 STEP 1: MEMBUAT 5 FILE PCAP SAMPLE

📝 Membuat file PCAP sample...
--------------------------------------------------

📁 Membuat PCAP01_23040700004.pcapng dengan 30 paket...
   ✅ PCAP01_23040700004.pcapng: 30 paket, 2.58 KB

📁 Membuat PCAP02_23040700004.pcapng dengan 50 paket...
   ✅ PCAP02_23040700004.pcapng: 50 paket, 4.28 KB

📁 Membuat PCAP03_23040700004.pcapng dengan 70 paket...
   ✅ PCAP03_23040700004.pcapng: 70 paket, 5.99 KB

📁 Membuat PCAP04_23040700004.pcapng dengan 90 paket...
   ✅ PCAP04_23040700004.pcapng: 90 paket, 7.69 KB

📁 Membuat PCAP05_23040700004.pcapng dengan 100 paket...
   ✅ PCAP05_23040700004.pcapng: 100 paket, 8.53 KB

--------------------------------------------------
✅ Berhasil membuat 5 file PCAP!

📊 STEP 2: PROSES FILE PCAP

📁 PCAP01_23040700004.pcapng
   📦 Paket: 30
   📊 Ukuran: 2.58 KB
   🔐 SHA-256: 64d1741a00ddcba63f6c17793c5b5b0280d6322

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ TUGAS SELESAI!
📁 File hasil: hasil_tugas.zip

📁 FILE YANG TERSEDIA:
   📄 PCAP01_23040700004.pcapng (2.58 KB)
   📄 PCAP02_23040700004.pcapng (4.28 KB)
   📄 PCAP03_23040700004.pcapng (5.99 KB)
   📄 PCAP04_23040700004.pcapng (7.69 KB)
   📄 PCAP05_23040700004.pcapng (8.53 KB)
